## WPI Turing setup

Training must run on a **GPU compute node**, not the login node. Docs: https://docs.turing.wpi.edu/

### 1. Access and clone

```bash
ssh <wpi-username>@turing.wpi.edu    # campus network or GlobalProtect VPN
git clone https://github.com/mpchouinard/DL_FinalProject.git
cd DL_FinalProject
```

### 2. Copy files not tracked in git (from local workstation)

| Path | What it is |
|------|------------|
| `Data/c_elegans_preprocessed.pkl` | Folds (from `ImportDataAndSetUpFolds.ipynb`) |
| `embeddings/receptors_650M.npz` | Cached ESM-2 receptor embeddings |
| `embeddings/peptides_650M.npz` | Cached ESM-2 peptide embeddings |
| `optuna_study.db` (optional) | Resume an interrupted Optuna study |

```bash
scp Data/c_elegans_preprocessed.pkl <user>@turing.wpi.edu:~/DL_FinalProject/Data/
scp embeddings/receptors_650M.npz embeddings/peptides_650M.npz <user>@turing.wpi.edu:~/DL_FinalProject/embeddings/
```

### 3. Python environment

Check current modules: https://docs.turing.wpi.edu/software/python/

```bash
module load python/3.11    # match the version listed in Turing docs
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
pip install jupyterlab ipywidgets    # needed for sjupyter
python -c "import torch; print('cuda:', torch.cuda.is_available())"
```

Install a **CUDA** build of PyTorch when `cuda: False` (see Turing Python docs; avoid CPU-only wheels).

### 4. Start Jupyter on a compute node

```bash
sjupyter
```

Or use **Open OnDemand → Jupyter** on https://ondemand.turing.wpi.edu (or the URL from Turing docs).

### 5. Notebook execution order

1. Setup cell, then all cells through **Training loop**.
2. **Parameter Sweep Optuna** (~15–30 min/trial on GPU; 40 trials ≈ 10–20 h).
3. `TOTAL_TRIALS` caps completed trials across runs; re-running the cell only fills in what's left (same budget as Deorpha `n_trials=40`).

In [11]:
# Setup for Colab, local, or WPI Turing; execute first.
# Local / Turing: install requirements.txt once; kernel cwd must be the repo root.
import os
import socket
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_TURING = "turing" in socket.gethostname().lower()

if ON_COLAB:
    # A fresh Colab VM is empty, so get the repo. Safe to re-run: it only clones if the repo isn't there yet.
    if not Path("Data").exists() and not Path("DL_FinalProject").exists():
        subprocess.run(["git", "clone", "https://github.com/mpchouinard/DL_FinalProject.git"], check=True)
    if Path("DL_FinalProject").exists():
        os.chdir("DL_FinalProject")
    # Colab has torch and scikit-learn, but its pandas is too old to read our pickle, and it lacks transformers.
    # Don't install requirements.txt here, it can replace Colab's GPU build of torch.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pandas", "transformers"], check=True)
    if "pandas" in sys.modules:
        print("pandas was already loaded; restart the session (Runtime > Restart session), then re-execute this cell.")

REQUIRED = [
    Path("Data/c_elegans_preprocessed.pkl"),
    Path("embeddings/receptors_650M.npz"),
    Path("embeddings/peptides_650M.npz"),
]
missing = [p for p in REQUIRED if not p.exists()]
if missing:
    lines = "\n  ".join(str(p) for p in missing)
    raise FileNotFoundError(
        f"Missing required files (copy from local workstation or run upstream notebooks):\n  {lines}\n"
        f"cwd: {os.getcwd()}"
    )

print("Colab:", ON_COLAB, "| Turing host:", ON_TURING, "| cwd:", os.getcwd())
for p in REQUIRED:
    print(f"  ok  {p}")

Colab: False | Turing host: False | cwd: C:\Users\natef\PycharmProjects\DL_FinalProject
  ok  Data\c_elegans_preprocessed.pkl
  ok  embeddings\receptors_650M.npz
  ok  embeddings\peptides_650M.npz


## Imports and data

In [12]:
import gc
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

from src.embed import load_embeddings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"device: cuda ({torch.cuda.get_device_name(0)})")
else:
    print("device: cpu — expect slow training; Turing requires sjupyter or a GPU node with CUDA PyTorch")

with open(Path("Data/c_elegans_preprocessed.pkl"), "rb") as f:
    data = pickle.load(f)
folds = data["folds"]
print(f"{len(folds)} folds loaded")

device: cpu — expect slow training; Turing requires sjupyter or a GPU node with CUDA PyTorch
10 folds loaded


## Model

In [13]:
import torch
from torch import nn


class CrossAttentionModel(nn.Module):
    """Predict whether a receptor-peptide pair is an agonist.

    Inputs are cached ESM-2 per-residue embeddings with shape
    (batch, padded_sequence_length, input_dim), plus true lengths.
    The 650M embeddings use input_dim=1280; attention runs in embedding_dim.
    """

    def __init__(self, input_dim=1280, embedding_dim=128, num_heads=4, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.receptor_proj = nn.Linear(input_dim, embedding_dim)
        self.peptide_proj = nn.Linear(input_dim, embedding_dim)
        self.receptor_norm = nn.LayerNorm(embedding_dim)
        self.peptide_norm = nn.LayerNorm(embedding_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attended_norm = nn.LayerNorm(embedding_dim)
        self.attended_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(3 * embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def masked_mean(values, padding_mask):
        # padding_mask is True where a row is padding.
        valid = (~padding_mask).unsqueeze(-1)
        return (values * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)

    def forward(self, receptor_residues, peptide_residues, receptor_lengths, peptide_lengths):
        # Cached embeddings are float16; match the model's parameter dtype.
        dtype = self.cross_attention.in_proj_weight.dtype
        receptor_residues = receptor_residues.to(dtype=dtype)
        peptide_residues = peptide_residues.to(dtype=dtype)
        receptor_lengths = torch.as_tensor(receptor_lengths, device=receptor_residues.device)
        peptide_lengths = torch.as_tensor(peptide_lengths, device=peptide_residues.device)

        receptor_padding = (
            torch.arange(receptor_residues.size(1), device=receptor_residues.device)[None, :]
            >= receptor_lengths[:, None]
        )
        peptide_padding = (
            torch.arange(peptide_residues.size(1), device=peptide_residues.device)[None, :]
            >= peptide_lengths[:, None]
        )

        receptor = self.receptor_norm(self.receptor_proj(receptor_residues))
        peptide = self.peptide_norm(self.peptide_proj(peptide_residues))

        # Each peptide residue queries the receptor residues. Padded receptor
        # positions cannot receive attention. Padded peptide positions are
        # excluded when we summarize the attended sequence below.
        attended, _ = self.cross_attention(
            query=peptide,
            key=receptor,
            value=receptor,
            key_padding_mask=receptor_padding,
            need_weights=False,
        )
        attended = self.attended_norm(peptide + self.attended_dropout(attended))

        pair_features = torch.cat(
            [
                self.masked_mean(peptide, peptide_padding),
                self.masked_mean(receptor, receptor_padding),
                self.masked_mean(attended, peptide_padding),
            ],
            dim=-1,
        )
        # One raw logit per pair; the training code will choose the loss.
        return self.classifier(pair_features).squeeze(-1)

## Encoder
Turns names into per-residue embeddings.

In [14]:
class CachedEncoder:
    """Looks up precomputed (frozen) ESM-2 embeddings by name."""

    def __init__(self, tag="650M", device=DEVICE, folder="embeddings"):
        self.device = device
        self.tables = {}
        for kind, file_prefix in [("receptor", "receptors"), ("peptide", "peptides")]:
            loaded = load_embeddings(f"{folder}/{file_prefix}_{tag}.npz")
            self.tables[kind] = {
                "row": {name: i for i, name in enumerate(loaded["names"])},
                "lengths": torch.as_tensor(loaded["lengths"], dtype=torch.long, device=device),
                "residues": torch.as_tensor(loaded["residues"], device=device),
            }

    def __call__(self, kind, names):
        table = self.tables[kind]
        rows = torch.as_tensor([table["row"][name] for name in names], device=self.device)
        lengths = table["lengths"][rows]
        longest = int(lengths.max())  # drop padding columns this batch doesn't need
        return table["residues"][rows, :longest], lengths


encoder = CachedEncoder("650M")

## Batching

In [15]:
def iterate_batches(pairs, batch_size, shuffle, generator=None):
    receptors = pairs["GPCR name"].to_numpy()
    peptides = pairs["Peptide"].to_numpy()
    labels = pairs["binds"].to_numpy().astype(np.float32)

    if shuffle:
        order = torch.randperm(len(pairs), generator=generator).numpy()
    else:
        order = np.arange(len(pairs))

    for start in range(0, len(pairs), batch_size):
        batch = order[start:start + batch_size]
        yield list(receptors[batch]), list(peptides[batch]), torch.as_tensor(labels[batch])


def forward_batch(model, encoder, receptor_names, peptide_names):
    receptor_residues, receptor_lengths = encoder("receptor", receptor_names)
    peptide_residues, peptide_lengths = encoder("peptide", peptide_names)
    return model(receptor_residues, peptide_residues, receptor_lengths, peptide_lengths)

## Losses

In [16]:
def bce_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)


def focal_loss(logits, targets, gamma=2.0, alpha=0.25):
    #BCE that down-weights easy pairs
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_true = torch.exp(-bce)
    class_weight = alpha * targets + (1 - alpha) * (1 - targets)
    return (class_weight * (1 - p_true) ** gamma * bce).mean()

## Evaluation
The main metric is per-receptor average precision, averaged over receptors (mAP), as in the paper.

In [17]:
def per_receptor_ap(pairs, scores):
    #average precision for each receptor that has at least one agonist
    scored = pairs.assign(score=scores)
    ap = {}
    for receptor, group in scored.groupby("GPCR name"):
        if group["binds"].any():
            ap[receptor] = average_precision_score(group["binds"], group["score"])
    return ap


@torch.no_grad()
def evaluate(model, encoder, pairs, batch_size=256):
    model.eval()
    scores = []
    for receptor_names, peptide_names, _ in iterate_batches(pairs, batch_size, shuffle=False):
        scores.append(forward_batch(model, encoder, receptor_names, peptide_names).float().cpu())
    scores = torch.cat(scores).numpy()

    labels = pairs["binds"].to_numpy().astype(int)
    receptor_ap = per_receptor_ap(pairs, scores)
    agonist_fraction = pairs.groupby("GPCR name")["binds"].mean()
    return {
        "map": np.mean(list(receptor_ap.values())),                       # mean of per-receptor AP
        "random_map": agonist_fraction[list(receptor_ap)].mean(),         # what a random ranking would get
        "receptor_ap": receptor_ap,
        "auprc": average_precision_score(labels, scores),                 # all pairs pooled
        "auroc": roc_auc_score(labels, scores),
        "n_agonists": int(labels.sum()),
    }

## Training loop

In [18]:
def sample_epoch_pairs(train_pairs, negatives_per_agonist, seed):
    #all agonists plus a fresh random sample of negatives
    agonists = train_pairs[train_pairs["binds"]]
    negatives = train_pairs[~train_pairs["binds"]]
    n_negatives = min(len(negatives), negatives_per_agonist * len(agonists))
    return pd.concat([agonists, negatives.sample(n_negatives, random_state=seed)])


def train_one_fold(fold, config, encoder, seed=0, fold_id=None):
    """Trains on the fold's training pairs and keeps the epoch with the best validation score.
    The test set is not used here. Returns the model, its saved weights and its validation scores."""

    if fold_id is not None:
        print(f"--- fold {fold_id}  hpt families: {fold['hpt_families']} ---")

    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = config["model_factory"]().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_score, best_epoch, best_weights, best_val = -1.0, -1, None, None
    for epoch in range(config["epochs"]):
        epoch_pairs = sample_epoch_pairs(fold["train"], config["negatives_per_agonist"], seed + epoch)

        model.train()
        total_loss, n_batches = 0.0, 0
        for receptor_names, peptide_names, labels in iterate_batches(
                epoch_pairs, config["batch_size"], shuffle=True, generator=generator):
            optimizer.zero_grad()
            logits = forward_batch(model, encoder, receptor_names, peptide_names)
            loss = config["loss_fn"](logits, labels.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        val = evaluate(model, encoder, fold["hyperparameter_tuning"])
        print(f"  epoch {epoch:2d}  train loss {total_loss / n_batches:.4f}  "
              f"val mAP {val['map']:.4f}  val AUPRC {val['auprc']:.4f}")

        if val[config["select_by"]] > best_score:
            best_score, best_epoch, best_val = val[config["select_by"]], epoch, val
            best_weights = {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}
        elif epoch - best_epoch >= config["patience"]:
            print(f"  early stop (best epoch {best_epoch}, val mAP {best_score:.4f})")
            break

    model.load_state_dict(best_weights)
    if fold_id is not None:
        print(f"  -> fold {fold_id} best val mAP {best_val['map']:.4f} (epoch {best_epoch})")
    return model, best_weights, best_val


def run_all_folds(folds, config, encoder, run_name, seed=0):
    """Trains every fold, scores its test set once, and saves each fold's weights to checkpoints/run_name/."""

    checkpoint_folder = Path("checkpoints") / run_name
    checkpoint_folder.mkdir(parents=True, exist_ok=True)

    rows = []
    for fold_id, fold in enumerate(folds):
        print(f"fold {fold_id}: test families {fold['test_families']}")
        model, weights, val = train_one_fold(fold, config, encoder, seed=seed)
        test = evaluate(model, encoder, fold["test"])
        print(f"  -> test mAP {test['map']:.4f} (random {test['random_map']:.4f})  "
              f"AUPRC {test['auprc']:.4f}  AUROC {test['auroc']:.4f}")

        rows.append({"fold": fold_id, "val_map": val["map"], "val_auprc": val["auprc"],
                     "test_map": test["map"], "test_random_map": test["random_map"],
                     "test_auprc": test["auprc"], "test_auroc": test["auroc"],
                     "test_agonists": test["n_agonists"], "receptor_ap": test["receptor_ap"]})
        torch.save(weights, checkpoint_folder / f"fold{fold_id}.pt")

        del model, weights  # free this fold's GPU memory before the next fold
        gc.collect()
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

## Parameter Sweep Optuna


In [ ]:
import optuna

SEED = 0
TOTAL_TRIALS = 40  # per fold; match DeorphaNN's inner Optuna budget
CHECKPOINT_FOLDER = Path("checkpoints") / "cross_attention_nested"
CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)

LOSS_FNS = {"bce": bce_loss, "focal": focal_loss}


def suggest_params(trial):
    return {
        "loss": trial.suggest_categorical("loss", ["bce", "focal"]),
        "dropout": trial.suggest_categorical("dropout", [0.1, 0.3, 0.5]),
        "lr": trial.suggest_float("lr", 3e-5, 3e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1, log=True),
        "negatives": trial.suggest_categorical("negatives", [4, 10]),
        "patience": trial.suggest_int("patience", 3, 8),
    }


def build_config(params):
    dropout = params["dropout"]
    return {
        "model_factory": lambda d=dropout: CrossAttentionModel(dropout=d),
        "lr": params["lr"],
        "weight_decay": params["weight_decay"],
        "negatives_per_agonist": params["negatives"],
        "loss_fn": LOSS_FNS[params["loss"]],
        "batch_size": 32,
        "epochs": 30,
        "patience": params["patience"],
        "select_by": "map",
        "grad_clip": 1.0,
    }


def make_objective(fold):
    def objective(trial):
        config = build_config(suggest_params(trial))
        _, _, val = train_one_fold(fold, config, encoder, seed=SEED)
        gc.collect()
        torch.cuda.empty_cache()
        return float(val["map"])
    return objective


def fold_for_refit(fold):
    # After HPs are locked, retrain on all non-test pairs (train ∪ HPT).
    # Epoch selection still uses the original HPT split.
    refit = dict(fold)
    refit["train"] = pd.concat(
        [fold["train"], fold["hyperparameter_tuning"]], ignore_index=True
    )
    return refit


nested_best = {}
nested_rows = []

for fold_id, fold in enumerate(folds):
    study_name = f"cross_attention_fold{fold_id}"
    print(f"\n=== fold {fold_id}  test {fold['test_families']}  hpt {fold['hpt_families']} ===")

    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        storage="sqlite:///optuna_study.db",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED + fold_id),
    )
    n_done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    remaining = max(0, TOTAL_TRIALS - n_done)
    if remaining == 0:
        print(f"  HPO already complete ({n_done}/{TOTAL_TRIALS})")
    else:
        print(f"  HPO {n_done}/{TOTAL_TRIALS}, running {remaining} more")
        study.optimize(make_objective(fold), n_trials=remaining, show_progress_bar=True)

    print(f"  best inner val mAP: {study.best_value:.4f}")
    print(f"  best params: {study.best_params}")
    nested_best[fold_id] = {"val_map": study.best_value, "params": study.best_params}

    config = build_config(study.best_params)
    model, weights, val = train_one_fold(
        fold_for_refit(fold), config, encoder, seed=SEED, fold_id=fold_id
    )
    test = evaluate(model, encoder, fold["test"])
    print(
        f"  -> test mAP {test['map']:.4f} (random {test['random_map']:.4f})  "
        f"AUPRC {test['auprc']:.4f}  AUROC {test['auroc']:.4f}"
    )
    torch.save(weights, CHECKPOINT_FOLDER / f"fold{fold_id}.pt")
    nested_rows.append({
        "fold": fold_id,
        "inner_val_map": study.best_value,
        "refit_val_map": val["map"],
        "test_map": test["map"],
        "test_random_map": test["random_map"],
        "test_auprc": test["auprc"],
        "test_auroc": test["auroc"],
        "test_agonists": test["n_agonists"],
        "receptor_ap": test["receptor_ap"],
        **study.best_params,
    })
    del model, weights
    gc.collect()
    torch.cuda.empty_cache()

nested_results = pd.DataFrame(nested_rows)
print("\nNested CV summary")
print(nested_results.drop(columns="receptor_ap").to_string(index=False))
print(nested_results[["inner_val_map", "test_map", "test_random_map", "test_auprc", "test_auroc"]].agg(["mean", "std"]))

[I 2026-11-24 00:41:51,233] A new study created in RDB with name: cross_attention_fold0



=== fold 0  test ['FRPR-16', 'FRPR-18', 'FRPR-4', 'FRPR-6', 'NPR-22', 'NMUR-2']  hpt ['TRHR-1', 'NPR-35', 'NPR-13', 'NPR-5', 'SEB-3'] ===
  HPO 0/40, running 40 more


  0%|          | 0/40 [00:00<?, ?it/s]

  epoch  0  train loss 0.0276  val mAP 0.0348  val AUPRC 0.0241
  epoch  1  train loss 0.0201  val mAP 0.0368  val AUPRC 0.0274
  epoch  2  train loss 0.0189  val mAP 0.0354  val AUPRC 0.0284
  epoch  3  train loss 0.0174  val mAP 0.0450  val AUPRC 0.0388
  epoch  4  train loss 0.0161  val mAP 0.0290  val AUPRC 0.0214
  epoch  5  train loss 0.0159  val mAP 0.0315  val AUPRC 0.0250
  epoch  6  train loss 0.0144  val mAP 0.0350  val AUPRC 0.0265
  epoch  7  train loss 0.0134  val mAP 0.0291  val AUPRC 0.0222
  epoch  8  train loss 0.0126  val mAP 0.0289  val AUPRC 0.0205
  early stop (best epoch 3, val mAP 0.0450)


Best trial: 0. Best value: 0.044966:   2%|▎         | 1/40 [03:14<2:06:18, 194.32s/it]

[I 2026-11-24 00:45:05,551] Trial 0 finished with value: 0.04496595468093917 and parameters: {'loss': 'focal', 'dropout': 0.1, 'lr': 0.00013274414292574905, 'weight_decay': 0.027389695952346985, 'negatives': 10, 'patience': 5}. Best is trial 0 with value: 0.04496595468093917.
  epoch  0  train loss 0.5421  val mAP 0.0307  val AUPRC 0.0224
  epoch  1  train loss 0.4293  val mAP 0.0343  val AUPRC 0.0250
  epoch  2  train loss 0.3833  val mAP 0.0357  val AUPRC 0.0251
  epoch  3  train loss 0.3667  val mAP 0.0361  val AUPRC 0.0251
  epoch  4  train loss 0.3669  val mAP 0.0368  val AUPRC 0.0253
  epoch  5  train loss 0.3290  val mAP 0.0409  val AUPRC 0.0263
  epoch  6  train loss 0.3188  val mAP 0.0425  val AUPRC 0.0274
  epoch  7  train loss 0.2971  val mAP 0.0490  val AUPRC 0.0314
  epoch  8  train loss 0.2857  val mAP 0.0468  val AUPRC 0.0394
  epoch  9  train loss 0.2875  val mAP 0.0605  val AUPRC 0.0531
  epoch 10  train loss 0.2783  val mAP 0.0442  val AUPRC 0.0325
  epoch 11  train l

... (6 outputs left out)

Best trial: 3. Best value: 0.124651:  12%|█▎        | 5/40 [16:53<2:03:29, 211.71s/it]

[I 2026-11-24 00:58:44,276] Trial 4 finished with value: 0.046380784709353806 and parameters: {'loss': 'focal', 'dropout': 0.1, 'lr': 0.00014052678947839518, 'weight_decay': 0.016232393459116295, 'negatives': 10, 'patience': 5}. Best is trial 3 with value: 0.12465081706344401.
  epoch  0  train loss 0.3332  val mAP 0.0335  val AUPRC 0.0242
  epoch  1  train loss 0.2355  val mAP 0.0348  val AUPRC 0.0240
  epoch  2  train loss 0.2070  val mAP 0.0352  val AUPRC 0.0247
  epoch  3  train loss 0.1922  val mAP 0.0430  val AUPRC 0.0272
  epoch  4  train loss 0.1819  val mAP 0.0479  val AUPRC 0.0343
  epoch  5  train loss 0.1759  val mAP 0.0504  val AUPRC 0.0418
  epoch  6  train loss 0.1701  val mAP 0.0561  val AUPRC 0.0473
  epoch  7  train loss 0.1600  val mAP 0.0561  val AUPRC 0.0495
  epoch  8  train loss 0.1556  val mAP 0.0418  val AUPRC 0.0353
  epoch  9  train loss 0.1491  val mAP 0.0487  val AUPRC 0.0425
  epoch 10  train loss 0.1481  val mAP 0.0625  val AUPRC 0.0608
  epoch 11  train 

Best trial: 3. Best value: 0.124651:  15%|█▌        | 6/40 [22:20<2:22:12, 250.94s/it]

[I 2026-11-24 01:04:11,378] Trial 5 finished with value: 0.06364168151099886 and parameters: {'loss': 'bce', 'dropout': 0.1, 'lr': 4.3494142458541404e-05, 'weight_decay': 0.044989205684622634, 'negatives': 10, 'patience': 4}. Best is trial 3 with value: 0.12465081706344401.


## Parameter sweep

In [ ]:
#Random search. Each trial trains on the four folds with the most validation agonists and records
#the average best validation mAP. The test sets are not used.
sweep_rows = []
for trial in range(15):
    lr = random.choice([3e-5, 1e-4, 3e-4])
    dropout = random.choice([0.1, 0.3, 0.5])
    weight_decay = random.choice([0.01, 0.1])
    negatives_per_agonist = random.choice([4, 10])
    loss_name = random.choice(["bce", "focal"])
    print(f"trial {trial}: lr {lr}, dropout {dropout}, weight decay {weight_decay}, "
          f"negatives {negatives_per_agonist}, loss {loss_name}")

    config = {
        "model_factory": lambda: CrossAttentionModel(dropout=dropout),
        "negatives_per_agonist": negatives_per_agonist,
        "loss_fn": bce_loss if loss_name == "bce" else focal_loss,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": 32,
        "epochs": 30,
        "patience": 5,
        "select_by": "map",
        "grad_clip": 1.0,
    }

    val_maps = []
    for fold_id in [5, 3, 8, 4]:
        model, weights, val = train_one_fold(folds[fold_id], config, encoder)
        val_maps.append(val["map"])
        del model, weights
        gc.collect()
        torch.cuda.empty_cache()

    sweep_rows.append({"lr": lr, "dropout": dropout, "weight_decay": weight_decay,
                       "negatives_per_agonist": negatives_per_agonist, "loss": loss_name,
                       "val_map": np.mean(val_maps)})
    pd.DataFrame(sweep_rows).to_csv("sweep_results.csv", index=False)  # saved after every trial
    print(f"  -> mean val mAP {np.mean(val_maps):.4f}")

print(pd.DataFrame(sweep_rows).sort_values("val_map", ascending=False).to_string(index=False))

In [ ]:
## Parameter Sweep Results
"""
trial 0: lr 0.0003, dropout 0.3, weight decay 0.01, negatives 10, loss focal
  epoch  0  train loss 0.0260  val mAP 0.1905  val AUPRC 0.3155
  epoch  1  train loss 0.0212  val mAP 0.1906  val AUPRC 0.3616
  epoch  2  train loss 0.0188  val mAP 0.2536  val AUPRC 0.3227
  epoch  3  train loss 0.0185  val mAP 0.1917  val AUPRC 0.2888
  epoch  4  train loss 0.0154  val mAP 0.2535  val AUPRC 0.3483
  epoch  0  train loss 0.0269  val mAP 0.2616  val AUPRC 0.2188
  epoch  1  train loss 0.0219  val mAP 0.2126  val AUPRC 0.3116
  epoch  2  train loss 0.0203  val mAP 0.2835  val AUPRC 0.2295
  epoch  3  train loss 0.0201  val mAP 0.2008  val AUPRC 0.2251
  epoch  4  train loss 0.0175  val mAP 0.1806  val AUPRC 0.2200
  epoch  0  train loss 0.0248  val mAP 0.0919  val AUPRC 0.0765
  epoch  1  train loss 0.0195  val mAP 0.1809  val AUPRC 0.1407
  epoch  2  train loss 0.0193  val mAP 0.0990  val AUPRC 0.1238
  epoch  3  train loss 0.0180  val mAP 0.1496  val AUPRC 0.0424
  epoch  4  train loss 0.0149  val mAP 0.0994  val AUPRC 0.2075
  epoch  0  train loss 0.0251  val mAP 0.0396  val AUPRC 0.0258
  epoch  1  train loss 0.0203  val mAP 0.1943  val AUPRC 0.2925
  epoch  2  train loss 0.0200  val mAP 0.2435  val AUPRC 0.2052
  epoch  3  train loss 0.0178  val mAP 0.1978  val AUPRC 0.2938
  epoch  4  train loss 0.0164  val mAP 0.1089  val AUPRC 0.1061
  -> mean val mAP 0.2404
trial 1: lr 3e-05, dropout 0.5, weight decay 0.1, negatives 4, loss focal
  epoch  0  train loss 0.0543  val mAP 0.1924  val AUPRC 0.3387
  epoch  1  train loss 0.0387  val mAP 0.1928  val AUPRC 0.3588
  epoch  2  train loss 0.0348  val mAP 0.1994  val AUPRC 0.3220
  epoch  3  train loss 0.0337  val mAP 0.1899  val AUPRC 0.3086
  epoch  4  train loss 0.0320  val mAP 0.1798  val AUPRC 0.2801
  epoch  0  train loss 0.0567  val mAP 0.3480  val AUPRC 0.4217
  epoch  1  train loss 0.0432  val mAP 0.2833  val AUPRC 0.4042
  epoch  2  train loss 0.0412  val mAP 0.2568  val AUPRC 0.3303
  epoch  3  train loss 0.0387  val mAP 0.2375  val AUPRC 0.2105
  epoch  4  train loss 0.0349  val mAP 0.2495  val AUPRC 0.1903
  epoch  0  train loss 0.0490  val mAP 0.0963  val AUPRC 0.0834
  epoch  1  train loss 0.0366  val mAP 0.1066  val AUPRC 0.0860
  epoch  2  train loss 0.0340  val mAP 0.1077  val AUPRC 0.1056
  epoch  3  train loss 0.0330  val mAP 0.1088  val AUPRC 0.1372
  epoch  4  train loss 0.0307  val mAP 0.1042  val AUPRC 0.1137
  epoch  0  train loss 0.0509  val mAP 0.2625  val AUPRC 0.2793
  epoch  1  train loss 0.0381  val mAP 0.2116  val AUPRC 0.2552
  epoch  2  train loss 0.0340  val mAP 0.2161  val AUPRC 0.2698
  epoch  3  train loss 0.0327  val mAP 0.2129  val AUPRC 0.2935
  epoch  4  train loss 0.0304  val mAP 0.1764  val AUPRC 0.3102
  -> mean val mAP 0.2297
trial 2: lr 0.0001, dropout 0.1, weight decay 0.01, negatives 4, loss focal
  epoch  0  train loss 0.0452  val mAP 0.2065  val AUPRC 0.3597
  epoch  1  train loss 0.0333  val mAP 0.1958  val AUPRC 0.3655
  epoch  2  train loss 0.0298  val mAP 0.2242  val AUPRC 0.3954
  epoch  3  train loss 0.0296  val mAP 0.1961  val AUPRC 0.3176
  epoch  4  train loss 0.0318  val mAP 0.2018  val AUPRC 0.3019
  epoch  0  train loss 0.0480  val mAP 0.2828  val AUPRC 0.2404
  epoch  1  train loss 0.0367  val mAP 0.2532  val AUPRC 0.2002
  epoch  2  train loss 0.0347  val mAP 0.2220  val AUPRC 0.1403
  epoch  3  train loss 0.0335  val mAP 0.1129  val AUPRC 0.0949
  epoch  4  train loss 0.0288  val mAP 0.2376  val AUPRC 0.1563
  epoch  0  train loss 0.0421  val mAP 0.1107  val AUPRC 0.0775
  epoch  1  train loss 0.0321  val mAP 0.1161  val AUPRC 0.1091
  epoch  2  train loss 0.0314  val mAP 0.1061  val AUPRC 0.1208
  epoch  3  train loss 0.0304  val mAP 0.1307  val AUPRC 0.1008
  epoch  4  train loss 0.0269  val mAP 0.1221  val AUPRC 0.0980
  epoch  0  train loss 0.0450  val mAP 0.2404  val AUPRC 0.3040
  epoch  1  train loss 0.0345  val mAP 0.1588  val AUPRC 0.2824
  epoch  2  train loss 0.0315  val mAP 0.1933  val AUPRC 0.3162
  epoch  3  train loss 0.0304  val mAP 0.1817  val AUPRC 0.3076
  epoch  4  train loss 0.0266  val mAP 0.2266  val AUPRC 0.3987
  -> mean val mAP 0.2195
trial 3: lr 0.0001, dropout 0.5, weight decay 0.1, negatives 10, loss focal
  epoch  0  train loss 0.0266  val mAP 0.1999  val AUPRC 0.3252
  epoch  1  train loss 0.0203  val mAP 0.1809  val AUPRC 0.2907
  epoch  2  train loss 0.0191  val mAP 0.1892  val AUPRC 0.3182
  epoch  3  train loss 0.0173  val mAP 0.1923  val AUPRC 0.3535
  epoch  4  train loss 0.0161  val mAP 0.2137  val AUPRC 0.2282
  epoch  0  train loss 0.0301  val mAP 0.2333  val AUPRC 0.2640
  epoch  1  train loss 0.0227  val mAP 0.2740  val AUPRC 0.2610
  epoch  2  train loss 0.0205  val mAP 0.1716  val AUPRC 0.1506
  epoch  3  train loss 0.0202  val mAP 0.2069  val AUPRC 0.2986
  epoch  4  train loss 0.0167  val mAP 0.1749  val AUPRC 0.1960
  epoch  0  train loss 0.0268  val mAP 0.1044  val AUPRC 0.0722
  epoch  1  train loss 0.0200  val mAP 0.1056  val AUPRC 0.1458
  epoch  2  train loss 0.0198  val mAP 0.1062  val AUPRC 0.1424
  epoch  3  train loss 0.0175  val mAP 0.1473  val AUPRC 0.1090
  epoch  4  train loss 0.0152  val mAP 0.1325  val AUPRC 0.2581
  epoch  0  train loss 0.0255  val mAP 0.2300  val AUPRC 0.3178
  epoch  1  train loss 0.0199  val mAP 0.1485  val AUPRC 0.2589
  epoch  2  train loss 0.0189  val mAP 0.2057  val AUPRC 0.1821
  epoch  3  train loss 0.0176  val mAP 0.1884  val AUPRC 0.3151
  epoch  4  train loss 0.0166  val mAP 0.1826  val AUPRC 0.2147
  -> mean val mAP 0.2163
trial 4: lr 3e-05, dropout 0.3, weight decay 0.1, negatives 4, loss bce
  epoch  0  train loss 0.4975  val mAP 0.1860  val AUPRC 0.3282
  epoch  1  train loss 0.3742  val mAP 0.1902  val AUPRC 0.3406
  epoch  2  train loss 0.3335  val mAP 0.1946  val AUPRC 0.3285
  epoch  3  train loss 0.3176  val mAP 0.1825  val AUPRC 0.3152
  epoch  4  train loss 0.3043  val mAP 0.1814  val AUPRC 0.3065
  epoch  0  train loss 0.5099  val mAP 0.3444  val AUPRC 0.4261
  epoch  1  train loss 0.4312  val mAP 0.2800  val AUPRC 0.3092
  epoch  2  train loss 0.3930  val mAP 0.2372  val AUPRC 0.2830
  epoch  3  train loss 0.3666  val mAP 0.2126  val AUPRC 0.1784
  epoch  4  train loss 0.3271  val mAP 0.2354  val AUPRC 0.1646
  epoch  0  train loss 0.4598  val mAP 0.0974  val AUPRC 0.0785
  epoch  1  train loss 0.3491  val mAP 0.1054  val AUPRC 0.0861
  epoch  2  train loss 0.3271  val mAP 0.1046  val AUPRC 0.1010
  epoch  3  train loss 0.3109  val mAP 0.1014  val AUPRC 0.1564
  epoch  4  train loss 0.2879  val mAP 0.1009  val AUPRC 0.1472
  epoch  0  train loss 0.4666  val mAP 0.2602  val AUPRC 0.2825
  epoch  1  train loss 0.3697  val mAP 0.2263  val AUPRC 0.2824
  epoch  2  train loss 0.3205  val mAP 0.1971  val AUPRC 0.2177
  epoch  3  train loss 0.3129  val mAP 0.1674  val AUPRC 0.2437
  epoch  4  train loss 0.2840  val mAP 0.1401  val AUPRC 0.2344
  -> mean val mAP 0.2261
trial 5: lr 0.0003, dropout 0.5, weight decay 0.01, negatives 10, loss focal
  epoch  0  train loss 0.0269  val mAP 0.1903  val AUPRC 0.3424
  epoch  1  train loss 0.0218  val mAP 0.1940  val AUPRC 0.4313
  epoch  2  train loss 0.0203  val mAP 0.2132  val AUPRC 0.2699
  epoch  3  train loss 0.0193  val mAP 0.2085  val AUPRC 0.4494
  epoch  4  train loss 0.0172  val mAP 0.2047  val AUPRC 0.3254
  epoch  0  train loss 0.0282  val mAP 0.3719  val AUPRC 0.3021
  epoch  1  train loss 0.0231  val mAP 0.3191  val AUPRC 0.2747
  epoch  2  train loss 0.0219  val mAP 0.2452  val AUPRC 0.1724
  epoch  3  train loss 0.0203  val mAP 0.1643  val AUPRC 0.1965
  epoch  4  train loss 0.0177  val mAP 0.2423  val AUPRC 0.2693
  epoch  0  train loss 0.0252  val mAP 0.0888  val AUPRC 0.0774
  epoch  1  train loss 0.0210  val mAP 0.2079  val AUPRC 0.1108
  epoch  2  train loss 0.0202  val mAP 0.1039  val AUPRC 0.0966
  epoch  3  train loss 0.0198  val mAP 0.1376  val AUPRC 0.0347
  epoch  4  train loss 0.0173  val mAP 0.1231  val AUPRC 0.0705
  epoch  0  train loss 0.0261  val mAP 0.2320  val AUPRC 0.3092
  epoch  1  train loss 0.0213  val mAP 0.1362  val AUPRC 0.2232
  epoch  2  train loss 0.0205  val mAP 0.1822  val AUPRC 0.1560
  epoch  3  train loss 0.0191  val mAP 0.2039  val AUPRC 0.3361
  epoch  4  train loss 0.0179  val mAP 0.1754  val AUPRC 0.2223
  -> mean val mAP 0.2563
trial 6: lr 3e-05, dropout 0.3, weight decay 0.01, negatives 10, loss bce
  epoch  0  train loss 0.2892  val mAP 0.1924  val AUPRC 0.3539
  epoch  1  train loss 0.2067  val mAP 0.1854  val AUPRC 0.2604
  epoch  2  train loss 0.1872  val mAP 0.1881  val AUPRC 0.2967
  epoch  3  train loss 0.1755  val mAP 0.1877  val AUPRC 0.2920
  epoch  4  train loss 0.1660  val mAP 0.1946  val AUPRC 0.2627
  epoch  0  train loss 0.3161  val mAP 0.2951  val AUPRC 0.3795
  epoch  1  train loss 0.2397  val mAP 0.2438  val AUPRC 0.2172
  epoch  2  train loss 0.2206  val mAP 0.1675  val AUPRC 0.1792
  epoch  3  train loss 0.2050  val mAP 0.1836  val AUPRC 0.1627
  epoch  4  train loss 0.1889  val mAP 0.1762  val AUPRC 0.1795
  epoch  0  train loss 0.2755  val mAP 0.0985  val AUPRC 0.0837
  epoch  1  train loss 0.2057  val mAP 0.1055  val AUPRC 0.0970
  epoch  2  train loss 0.1957  val mAP 0.0980  val AUPRC 0.2223
  epoch  3  train loss 0.1854  val mAP 0.0976  val AUPRC 0.1680
  epoch  4  train loss 0.1681  val mAP 0.1038  val AUPRC 0.1052
  epoch  0  train loss 0.2829  val mAP 0.2568  val AUPRC 0.2784
  epoch  1  train loss 0.2068  val mAP 0.2489  val AUPRC 0.2765
  epoch  2  train loss 0.1907  val mAP 0.1661  val AUPRC 0.1732
  epoch  3  train loss 0.1822  val mAP 0.1788  val AUPRC 0.2910
  epoch  4  train loss 0.1676  val mAP 0.2022  val AUPRC 0.2979
  -> mean val mAP 0.2130
trial 7: lr 0.0001, dropout 0.3, weight decay 0.1, negatives 4, loss bce
  epoch  0  train loss 0.4340  val mAP 0.2007  val AUPRC 0.3741
  epoch  1  train loss 0.3283  val mAP 0.1798  val AUPRC 0.3497
  epoch  2  train loss 0.2855  val mAP 0.2114  val AUPRC 0.3374
  epoch  3  train loss 0.2881  val mAP 0.1792  val AUPRC 0.2660
  epoch  4  train loss 0.2765  val mAP 0.2006  val AUPRC 0.3335
  epoch  0  train loss 0.4789  val mAP 0.2801  val AUPRC 0.2678
  epoch  1  train loss 0.3831  val mAP 0.1897  val AUPRC 0.1718
  epoch  2  train loss 0.3580  val mAP 0.1672  val AUPRC 0.1376
  epoch  3  train loss 0.3334  val mAP 0.1933  val AUPRC 0.1727
  epoch  4  train loss 0.2881  val mAP 0.2298  val AUPRC 0.1676
  epoch  0  train loss 0.3985  val mAP 0.1079  val AUPRC 0.0778
  epoch  1  train loss 0.3229  val mAP 0.1065  val AUPRC 0.2175
  epoch  2  train loss 0.3029  val mAP 0.1029  val AUPRC 0.1002
  epoch  3  train loss 0.2802  val mAP 0.1159  val AUPRC 0.0842
  epoch  4  train loss 0.2531  val mAP 0.0965  val AUPRC 0.0775
  epoch  0  train loss 0.4269  val mAP 0.2255  val AUPRC 0.2755
  epoch  1  train loss 0.3440  val mAP 0.1429  val AUPRC 0.2516
  epoch  2  train loss 0.2997  val mAP 0.1376  val AUPRC 0.2009
  epoch  3  train loss 0.2870  val mAP 0.1345  val AUPRC 0.1884
  epoch  4  train loss 0.2510  val mAP 0.1516  val AUPRC 0.1724
  -> mean val mAP 0.2082
trial 8: lr 3e-05, dropout 0.3, weight decay 0.1, negatives 10, loss bce
  epoch  0  train loss 0.2892  val mAP 0.1924  val AUPRC 0.3539
  epoch  1  train loss 0.2067  val mAP 0.1854  val AUPRC 0.2604
  epoch  2  train loss 0.1873  val mAP 0.1879  val AUPRC 0.2967
  epoch  3  train loss 0.1755  val mAP 0.1877  val AUPRC 0.2921
  epoch  4  train loss 0.1661  val mAP 0.1946  val AUPRC 0.2630
  epoch  0  train loss 0.3161  val mAP 0.2951  val AUPRC 0.3789
  epoch  1  train loss 0.2397  val mAP 0.2438  val AUPRC 0.2174
  epoch  2  train loss 0.2206  val mAP 0.1675  val AUPRC 0.1791
  epoch  3  train loss 0.2050  val mAP 0.1835  val AUPRC 0.1627
  epoch  4  train loss 0.1889  val mAP 0.1763  val AUPRC 0.1794
  epoch  0  train loss 0.2755  val mAP 0.0985  val AUPRC 0.0837
  epoch  1  train loss 0.2057  val mAP 0.1055  val AUPRC 0.0970
  epoch  2  train loss 0.1958  val mAP 0.0980  val AUPRC 0.2232
  epoch  3  train loss 0.1855  val mAP 0.0976  val AUPRC 0.1699
  epoch  4  train loss 0.1681  val mAP 0.1032  val AUPRC 0.1089
  epoch  0  train loss 0.2829  val mAP 0.2568  val AUPRC 0.2784
  epoch  1  train loss 0.2068  val mAP 0.2490  val AUPRC 0.2766
  epoch  2  train loss 0.1907  val mAP 0.1668  val AUPRC 0.1734
  epoch  3  train loss 0.1822  val mAP 0.1806  val AUPRC 0.2921
  epoch  4  train loss 0.1677  val mAP 0.2022  val AUPRC 0.2979
  -> mean val mAP 0.2130
trial 9: lr 3e-05, dropout 0.1, weight decay 0.01, negatives 10, loss bce
  epoch  0  train loss 0.2858  val mAP 0.1953  val AUPRC 0.3491
  epoch  1  train loss 0.2041  val mAP 0.1849  val AUPRC 0.2585
  epoch  2  train loss 0.1862  val mAP 0.1884  val AUPRC 0.2844
  epoch  3  train loss 0.1733  val mAP 0.1902  val AUPRC 0.2859
  epoch  4  train loss 0.1597  val mAP 0.2055  val AUPRC 0.2501
  epoch  0  train loss 0.3144  val mAP 0.2869  val AUPRC 0.3818
  epoch  1  train loss 0.2372  val mAP 0.2395  val AUPRC 0.2282
  epoch  2  train loss 0.2175  val mAP 0.2078  val AUPRC 0.1862
  epoch  3  train loss 0.2014  val mAP 0.2004  val AUPRC 0.1702
  epoch  4  train loss 0.1846  val mAP 0.1740  val AUPRC 0.1831
  epoch  0  train loss 0.2724  val mAP 0.0992  val AUPRC 0.0857
  epoch  1  train loss 0.2032  val mAP 0.1058  val AUPRC 0.1188
  epoch  2  train loss 0.1921  val mAP 0.0960  val AUPRC 0.2818
  epoch  3  train loss 0.1821  val mAP 0.1004  val AUPRC 0.2427
  epoch  4  train loss 0.1665  val mAP 0.1110  val AUPRC 0.1466
  epoch  0  train loss 0.2785  val mAP 0.2564  val AUPRC 0.2803
  epoch  1  train loss 0.2059  val mAP 0.2386  val AUPRC 0.2662
  epoch  2  train loss 0.1887  val mAP 0.1593  val AUPRC 0.1891
  epoch  3  train loss 0.1786  val mAP 0.1811  val AUPRC 0.3049
  epoch  4  train loss 0.1650  val mAP 0.2078  val AUPRC 0.3217
  -> mean val mAP 0.2149
trial 10: lr 3e-05, dropout 0.1, weight decay 0.1, negatives 4, loss focal
  epoch  0  train loss 0.0512  val mAP 0.1964  val AUPRC 0.3574
  epoch  1  train loss 0.0369  val mAP 0.1948  val AUPRC 0.3586
  epoch  2  train loss 0.0336  val mAP 0.1986  val AUPRC 0.3177
  epoch  3  train loss 0.0322  val mAP 0.1854  val AUPRC 0.3103
  epoch  4  train loss 0.0313  val mAP 0.1820  val AUPRC 0.2824
  epoch  0  train loss 0.0558  val mAP 0.3431  val AUPRC 0.4082
  epoch  1  train loss 0.0419  val mAP 0.2734  val AUPRC 0.3944
  epoch  2  train loss 0.0392  val mAP 0.2320  val AUPRC 0.2644
  epoch  3  train loss 0.0367  val mAP 0.2023  val AUPRC 0.1718
  epoch  4  train loss 0.0331  val mAP 0.2383  val AUPRC 0.1596
  epoch  0  train loss 0.0473  val mAP 0.0988  val AUPRC 0.0845
  epoch  1  train loss 0.0347  val mAP 0.1120  val AUPRC 0.0974
  epoch  2  train loss 0.0333  val mAP 0.1095  val AUPRC 0.1149
  epoch  3  train loss 0.0319  val mAP 0.1096  val AUPRC 0.1570
  epoch  4  train loss 0.0295  val mAP 0.1165  val AUPRC 0.2321
  epoch  0  train loss 0.0491  val mAP 0.2619  val AUPRC 0.2931
  epoch  1  train loss 0.0367  val mAP 0.2031  val AUPRC 0.2610
  epoch  2  train loss 0.0321  val mAP 0.2098  val AUPRC 0.2806
  epoch  3  train loss 0.0313  val mAP 0.1826  val AUPRC 0.2933
  epoch  4  train loss 0.0285  val mAP 0.2127  val AUPRC 0.3538
  -> mean val mAP 0.2300
trial 11: lr 0.0003, dropout 0.3, weight decay 0.1, negatives 4, loss focal
  epoch  0  train loss 0.0407  val mAP 0.2025  val AUPRC 0.2615
  epoch  1  train loss 0.0336  val mAP 0.2096  val AUPRC 0.3951
  epoch  2  train loss 0.0286  val mAP 0.2238  val AUPRC 0.4024
  epoch  3  train loss 0.0316  val mAP 0.1651  val AUPRC 0.0754
  epoch  4  train loss 0.0337  val mAP 0.2496  val AUPRC 0.5374
  epoch  0  train loss 0.0503  val mAP 0.3111  val AUPRC 0.3592
  epoch  1  train loss 0.0371  val mAP 0.3380  val AUPRC 0.3560
  epoch  2  train loss 0.0346  val mAP 0.2876  val AUPRC 0.2786
  epoch  3  train loss 0.0334  val mAP 0.1582  val AUPRC 0.2352
  epoch  4  train loss 0.0276  val mAP 0.2040  val AUPRC 0.2200
  epoch  0  train loss 0.0407  val mAP 0.1161  val AUPRC 0.0693
  epoch  1  train loss 0.0334  val mAP 0.1255  val AUPRC 0.0900
  epoch  2  train loss 0.0299  val mAP 0.1034  val AUPRC 0.0367
  epoch  3  train loss 0.0302  val mAP 0.0991  val AUPRC 0.0405
  epoch  4  train loss 0.0248  val mAP 0.0687  val AUPRC 0.0401
  epoch  0  train loss 0.0408  val mAP 0.1536  val AUPRC 0.2629
  epoch  1  train loss 0.0303  val mAP 0.1716  val AUPRC 0.1205
  epoch  2  train loss 0.0296  val mAP 0.1817  val AUPRC 0.2521
  epoch  3  train loss 0.0292  val mAP 0.1667  val AUPRC 0.2723
  epoch  4  train loss 0.0245  val mAP 0.1919  val AUPRC 0.2816
  -> mean val mAP 0.2262
trial 12: lr 0.0003, dropout 0.5, weight decay 0.1, negatives 10, loss focal
  epoch  0  train loss 0.0270  val mAP 0.1889  val AUPRC 0.3937
  epoch  1  train loss 0.0218  val mAP 0.1861  val AUPRC 0.4079
  epoch  2  train loss 0.0197  val mAP 0.1986  val AUPRC 0.2197
  epoch  3  train loss 0.0190  val mAP 0.2268  val AUPRC 0.4390
  epoch  4  train loss 0.0172  val mAP 0.3051  val AUPRC 0.3433
  epoch  0  train loss 0.0282  val mAP 0.3790  val AUPRC 0.3105
  epoch  1  train loss 0.0232  val mAP 0.3148  val AUPRC 0.3875
  epoch  2  train loss 0.0222  val mAP 0.2726  val AUPRC 0.2677
  epoch  3  train loss 0.0205  val mAP 0.3023  val AUPRC 0.2704
  epoch  4  train loss 0.0187  val mAP 0.3202  val AUPRC 0.2975
  epoch  0  train loss 0.0252  val mAP 0.0998  val AUPRC 0.0851
  epoch  1  train loss 0.0211  val mAP 0.1620  val AUPRC 0.0676
  epoch  2  train loss 0.0201  val mAP 0.0927  val AUPRC 0.0530
  epoch  3  train loss 0.0190  val mAP 0.0923  val AUPRC 0.0343
  epoch  4  train loss 0.0161  val mAP 0.1091  val AUPRC 0.0490
  epoch  0  train loss 0.0261  val mAP 0.2359  val AUPRC 0.3019
  epoch  1  train loss 0.0212  val mAP 0.1388  val AUPRC 0.2487
  epoch  2  train loss 0.0208  val mAP 0.1077  val AUPRC 0.0498
  epoch  3  train loss 0.0194  val mAP 0.2093  val AUPRC 0.3454
  epoch  4  train loss 0.0182  val mAP 0.2385  val AUPRC 0.1599
  -> mean val mAP 0.2711
trial 13: lr 0.0003, dropout 0.5, weight decay 0.01, negatives 4, loss bce
  epoch  0  train loss 0.4212  val mAP 0.1880  val AUPRC 0.3690
  epoch  1  train loss 0.3224  val mAP 0.1857  val AUPRC 0.3046
  epoch  2  train loss 0.2939  val mAP 0.1942  val AUPRC 0.2515
  epoch  3  train loss 0.2719  val mAP 0.1953  val AUPRC 0.2916
  epoch  4  train loss 0.2966  val mAP 0.2039  val AUPRC 0.3754
  epoch  0  train loss 0.4795  val mAP 0.2957  val AUPRC 0.2354
  epoch  1  train loss 0.3725  val mAP 0.2520  val AUPRC 0.1808
  epoch  2  train loss 0.3359  val mAP 0.2901  val AUPRC 0.2482
  epoch  3  train loss 0.3471  val mAP 0.1883  val AUPRC 0.1221
  epoch  4  train loss 0.3019  val mAP 0.1169  val AUPRC 0.0630
  epoch  0  train loss 0.3868  val mAP 0.1048  val AUPRC 0.0566
  epoch  1  train loss 0.3226  val mAP 0.0946  val AUPRC 0.0339
  epoch  2  train loss 0.2892  val mAP 0.1293  val AUPRC 0.0648
  epoch  3  train loss 0.2828  val mAP 0.1442  val AUPRC 0.0675
  epoch  4  train loss 0.2545  val mAP 0.1049  val AUPRC 0.0610
  epoch  0  train loss 0.3905  val mAP 0.1943  val AUPRC 0.1247
  epoch  1  train loss 0.3592  val mAP 0.2144  val AUPRC 0.2614
  epoch  2  train loss 0.3095  val mAP 0.1284  val AUPRC 0.0487
  epoch  3  train loss 0.3007  val mAP 0.1859  val AUPRC 0.2635
  epoch  4  train loss 0.2696  val mAP 0.1952  val AUPRC 0.0556
  -> mean val mAP 0.2146
trial 14: lr 0.0001, dropout 0.1, weight decay 0.1, negatives 4, loss bce
  epoch  0  train loss 0.4267  val mAP 0.2006  val AUPRC 0.3404
  epoch  1  train loss 0.3262  val mAP 0.1811  val AUPRC 0.3233
  epoch  2  train loss 0.2828  val mAP 0.2116  val AUPRC 0.3215
  epoch  3  train loss 0.2807  val mAP 0.1835  val AUPRC 0.2863
  epoch  4  train loss 0.2767  val mAP 0.2209  val AUPRC 0.3381
  epoch  0  train loss 0.4706  val mAP 0.2720  val AUPRC 0.2400
  epoch  1  train loss 0.3754  val mAP 0.2247  val AUPRC 0.1759
  epoch  2  train loss 0.3549  val mAP 0.2294  val AUPRC 0.1551
  epoch  3  train loss 0.3354  val mAP 0.2378  val AUPRC 0.2203
  epoch  4  train loss 0.2872  val mAP 0.2667  val AUPRC 0.1905
  epoch  0  train loss 0.3940  val mAP 0.1086  val AUPRC 0.0800
  epoch  1  train loss 0.3126  val mAP 0.1051  val AUPRC 0.1819
  epoch  2  train loss 0.2976  val mAP 0.1075  val AUPRC 0.1015
  epoch  3  train loss 0.2760  val mAP 0.1214  val AUPRC 0.0881
  epoch  4  train loss 0.2463  val mAP 0.1143  val AUPRC 0.0823
  epoch  0  train loss 0.4253  val mAP 0.2460  val AUPRC 0.2846
  epoch  1  train loss 0.3463  val mAP 0.1609  val AUPRC 0.2758
  epoch  2  train loss 0.2915  val mAP 0.1443  val AUPRC 0.1558
  epoch  3  train loss 0.2836  val mAP 0.1051  val AUPRC 0.1430
  epoch  4  train loss 0.2548  val mAP 0.1311  val AUPRC 0.1557
  -> mean val mAP 0.2151
     lr  dropout  weight_decay  negatives_per_agonist  loss  val_map
0.00030      0.5          0.10                     10 focal 0.271143
0.00030      0.5          0.01                     10 focal 0.256259
0.00030      0.3          0.01                     10 focal 0.240391
0.00003      0.1          0.10                      4 focal 0.230025
0.00003      0.5          0.10                      4 focal 0.229690
0.00030      0.3          0.10                      4 focal 0.226236
0.00003      0.3          0.10                      4   bce 0.226137
0.00010      0.1          0.01                      4 focal 0.219534
0.00010      0.5          0.10                     10 focal 0.216257
0.00010      0.1          0.10                      4   bce 0.215083
0.00003      0.1          0.01                     10   bce 0.214948
0.00030      0.5          0.01                      4   bce 0.214561
0.00003      0.3          0.01                     10   bce 0.213037
0.00003      0.3          0.10                     10   bce 0.213027
0.00010      0.3          0.10                      4   bce 0.208222
"""

## Final run

In [ ]:
config = {
    "model_factory": lambda: CrossAttentionModel(dropout=0.1),
    "negatives_per_agonist": 4,
    "loss_fn": bce_loss,        # or focal_loss
    "lr": 1e-4,
    "weight_decay": 0.01,
    "batch_size": 32,
    "epochs": 30,
    "patience": 5,
    "select_by": "map",         # validation score that picks the best epoch: "map" or "auprc"
    "grad_clip": 1.0,
}

results = run_all_folds(folds, config, encoder, run_name="cross_attention_bce")
print(results.drop(columns="receptor_ap").to_string(index=False))
print(results[["test_map", "test_random_map", "test_auprc", "test_auroc"]].agg(["mean", "std"]))